
# Regression Dataset Tutorial: GNN Training

This notebook demonstrates how to set up and train a Graph Neural Network (GNN) using PyTorch and PyTorch Geometric. You will learn how to prepare data, define a model, and train and evaluate it on a dataset. Let's get started!


In [ ]:
import torch
from torch_geometric.loader import DataLoader
from copy import deepcopy

# Imports Tutorials
from hg_models import HeteroGNN_SAGE
from early_stopping import EarlyStopping
from relsc_h import RelSCH
from relsc_m import RelSCM


## Loading the Dataset

In this section, we load various datasets using the `RelSCM` class. After that we can see the number of nodes os `relsc_m_dataset1` and the structure of one of his nodes


In [2]:
relsc_m_dataset1 = RelSCM(root="./data", project_name="rdf")
#relsc_m_dataset2 = RelSCM(root="./data", project_name="dubbo")
#relsc_m_dataset3 = RelSCM(root="./data", project_name="H2")
#relsc_m_dataset4 = RelSCM(root="./data", project_name="hadoop")
#relsc_m_dataset5 = RelSCM(root="./data", project_name="systemds")
#relsc_m_dataset6 = RelSCM(root="./data", project_name="ossbuilds")

Files for rdf already exist in data/RelSC/raw. Skipping download.


In [3]:
relsc_m_dataset1

RelSCM(478)

In [4]:
relsc_m_dataset1[0]

HeteroData(
  y=[1],
  code_structure={ x=[8, 84] },
  literals_and_constants={ x=[24, 84] },
  declarations={ x=[8, 84] },
  types_and_references={ x=[5, 84] },
  control_flow={ x=[1, 84] },
  expressions_and_operations={ x=[2, 84] },
  (code_structure, 0, code_structure)={ edge_index=[2, 4] }
)

In [5]:
# Training and testing functions for the model
def train(model, data_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for data in data_loader:
        data = data.to(device)
        optimizer.zero_grad()
        graph_output = model(data.x_dict, data.edge_index_dict, data.batch_dict)
        target = data.y.view(-1, 1).to(device)
        loss = criterion(graph_output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)

@torch.no_grad()
def test(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    preds, reals = [], []
    for data in loader:
        data = data.to(device)
        
        out = model(data.x_dict, data.edge_index_dict, data.batch_dict)
        loss = criterion(out, torch.reshape(data.y, (data.y.shape[0], 1)))
        total_loss += float(loss)

        pred = out.detach().cpu().numpy()[:, 0] if out.dim() > 1 else out.detach().cpu().numpy()
        real = data.y.detach().cpu().numpy()
        preds.extend(pred)
        reals.extend(real)

    return total_loss / len(loader), 0, preds, reals


## Defining Hyperparameters

Here we set the model's hyperparameters like the device (CPU or GPU), patience (for early stopping), hidden channels, batch size, and learning rate. You can adjust these parameters based on your computational resources and dataset size.


In [6]:
# Defining the device, patience, hidden channels, batch size, and learning rate
device = "cpu" # cuda, mps
patience = 15
hidden_channels = 32
batch_size = 32
learning_rate = 0.01

## Training the Model

In this section, we define the model, optimizer, and loss criterion, and set up the data loaders for training, validation, and testing. We use a special `collate_function` which is available in all the `relsc_m` datasets (normally the default one works just fine). The model is trained using the Adam optimizer and `L1Loss` as the loss function. We also use early stopping to prevent overfitting by monitoring the validation loss. The best model based on validation loss is saved, and training progress is printed at regular intervals.


In [7]:
model = HeteroGNN_SAGE(relsc_m_dataset1, hidden_channels=hidden_channels, out_channels=1, num_layers=4, device=device).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.L1Loss()

In [8]:
# Training and testing functions for the model
split_idx = relsc_m_dataset1.get_idx_split()

train_loader = DataLoader(relsc_m_dataset1[split_idx['train']], batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(relsc_m_dataset1[split_idx['val']],   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(relsc_m_dataset1[split_idx['test']],  batch_size=batch_size, shuffle=False)

In [9]:
# Training and testing functions for the model
early_stopping = EarlyStopping(patience=patience)
best_model_state = None
best_val_loss = float('inf')

for epoch in range(1, 101):
    # Train the model
    train_loss = train(model, train_loader, optimizer, criterion, device)

    # Validate the model
    val_loss, _, _, _ = test(model, val_loader, criterion, device)

    # Save the best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = deepcopy(model.state_dict())
    
    # Print progress every 10 epochs or at the first epoch
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, Val Loss: {val_loss:.3f}')

    # Early stopping check based on validation loss
    if early_stopping(val_loss):
        print(f'Early stopping at epoch {epoch}')
        break

Epoch: 001, Train Loss: 0.376, Val Loss: 0.153
Epoch: 010, Train Loss: 0.152, Val Loss: 0.154
Epoch: 020, Train Loss: 0.155, Val Loss: 0.148
Epoch: 030, Train Loss: 0.152, Val Loss: 0.151
Epoch: 040, Train Loss: 0.153, Val Loss: 0.153
Early stopping triggered after 15 epochs of no improvement.
Early stopping at epoch 47



## Testing the Model

The `test` function evaluates the model's performance on the validation or test dataset. It runs in evaluation mode, calculating the loss without performing backpropagation, and collects predictions for further analysis.


In [10]:
# Training and testing functions for the model
# Reload the best model after training
model.load_state_dict(best_model_state)

# After training completes, evaluate the model on the test set
test_loss, _, predt, realt = test(model, test_loader, criterion, device)
val_loss, _, predt, realt = test(model, val_loader, criterion, device)

# Print final test results
print(f'Final Test Loss: {test_loss:.3f}, Final val Loss: {val_loss:.3f}')

Final Test Loss: 0.124, Final val Loss: 0.141


In [11]:
# Same thing with Homogeneus Loading the RelSCH datasets
type1_dataset1 = RelSCH("data", project_name="rdf")
#type1_dataset2 = RelSCH("data", project_name="dubbo")
#type1_dataset3 = RelSCH("data", project_name="H2")
#type1_dataset4 = RelSCH("data", project_name="hadoop")
#type1_dataset5 = RelSCH("data", project_name="systemds")
#type1_dataset6 = RelSCH("data", project_name="ossbuilds")

Files for rdf already exist in data/RelSC/raw. Skipping download.
